In [ ]:
import struct
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. DEFINE ARCHITECTURE & HYPERPARAMETERS
# ==========================================
# Topology: 1 input (x), two hidden layers with 16 neurons, 1 output (sin(x))
LAYER_SIZES = [1, 16, 16, 1]
IS_REGRESSION = True  # True = Regression for continuous sine wave output

model = models.Sequential()

# Input layer + First hidden layer
model.add(layers.Dense(LAYER_SIZES[1], activation='relu', input_shape=(LAYER_SIZES[0],)))

# Intermediate hidden layers
for size in LAYER_SIZES[2:-1]:
    model.add(layers.Dense(size, activation='relu'))

# Output layer (Linear activation for continuous values)
model.add(layers.Dense(LAYER_SIZES[-1], activation='linear'))

# ==========================================
# 2. GENERATE SINE WAVE DATA & TRAIN
# ==========================================
# Generate 1000 sample points between -pi and +pi
X_train = np.linspace(-np.pi, np.pi, 1000, dtype=np.float32).reshape(-1, 1)
y_train = np.sin(X_train).astype(np.float32)

# Compile model using Adam for faster convergence on function approximation
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse')

print("Training MLP to approximate sine wave...")
model.fit(X_train, y_train, epochs=500, batch_size=32, verbose=1)

# ==========================================
# 3. EXPORT BINARY FOR MICROPYTHON
# ==========================================
def export_microml_bin(model: tf.keras.Model, layer_sizes: list, is_regression: bool, filepath: str):
    with open(filepath, "wb") as f:
        num_layers = len(layer_sizes)
        is_regression_int = 1 if is_regression else 0

        # 1. Header: num_layers (int32), is_regression (int32)
        f.write(struct.pack("<ii", num_layers, is_regression_int))

        # 2. Layer sizes: array of int32
        f.write(struct.pack(f"<{num_layers}i", *layer_sizes))

        # 3. Weights and Biases (Layer by Layer)
        for layer in model.layers:
            weights, biases = layer.get_weights()

            # Keras [in_dim, out_dim] order matches C row-major layout
            weights_bin = weights.astype('<f4').tobytes()
            biases_bin = biases.astype('<f4').tobytes()

            f.write(weights_bin)
            f.write(biases_bin)

    print(f"\nModel successfully saved to '{filepath}'")

export_microml_bin(model, LAYER_SIZES, IS_REGRESSION, "sine_mlp.bin")

Training MLP to approximate sine wave...
Epoch 1/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1652
Epoch 2/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0861
Epoch 3/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0304
Epoch 4/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0071
Epoch 5/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0021
Epoch 6/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0010
Epoch 7/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.2400e-04
Epoch 8/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.1870e-04
Epoch 9/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.4312e-04
Epoch 10/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.8813e-04
Epoch 11/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.5860e-04
Epoch 12/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6489e-04
Epoch 13/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2658e-04
Epoch 14/2000
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - l